# **HealthConnect Experience Lab — Week 4**
## **Machine Learning Problem Definition and Initial Data Assessment**

**AnalystLab Africa | Data Science Track | Shylet Nyazika**



**Week 4 scope:** Understand the business problem, assess data suitability, define the target and candidate features, and propose an initial modelling approach. A final model is not required at this stage.

In [3]:
from google.colab import files
uploaded = files.upload()

Saving HealthConnect_Appointment_Data.csv to HealthConnect_Appointment_Data (1).csv


In [4]:
import pandas as pd
import numpy as np

df = pd.read_csv("HealthConnect_Appointment_Data.csv")
print("Shape:", df.shape)
df.head()

Shape: (5000, 18)


,appointment_id,patient_id,gender,age,age_group,appointment_type,booking_date,appointment_date,appointment_day,appointment_time,booking_lead_days,previous_appointments,previous_no_shows,reminder_sent,reminder_channel,distance_to_clinic_km,waiting_time_minutes,appointment_outcome
0,HC-00001,P-1613,Female,39,35-44,Follow-up,2/6/2025,2/18/2025,Tuesday,Afternoon,12,2,0,Yes,WhatsApp,19.3,29.0,No-Show
1,HC-00002,P-0813,Male,31,25-34,Specialist Consultation,2/25/2026,2/27/2026,Friday,Morning,2,6,0,Yes,SMS,14.3,42.0,Attended
2,HC-00003,P-1366,Female,50,45-54,General Consultation,11/16/2025,12/24/2025,Wednesday,Morning,38,5,1,Yes,SMS,11.4,11.0,No-Show
3,HC-00004,P-1031,Male,59,55-64,Follow-up,7/18/2025,8/28/2025,Thursday,Evening,41,3,1,Yes,SMS,7.4,35.0,Attended
4,HC-00005,P-1458,Female,34,25-34,Follow-up,7/9/2025,8/25/2025,Monday,Afternoon,47,3,1,Yes,Email,5.6,27.0,No-Show


# Part 1: Project and Resource Review


**Problem, Role, and Approach**

**The business problem:** HealthConnect Clinic experiences missed appointments (no-shows), which wastes appointment slots and reduces care efficiency. The clinic wants to understand which factors are associated with no-shows and explore whether this can be predicted in advance.

**My role (Data Science track):** Define the machine learning problem precisely, assess whether the available data can realistically support a no-show prediction model, and propose an initial modelling approach — without yet building or training a model.

**Resources I'm using:** The HealthConnect Appointment Dataset (5,000 records, 18 variables) and the accompanying Data Dictionary, both provided by AnalystLab Africa for this project.

# **Part 2 (Track Task): Initial Data Assessment**

In [5]:
summary = pd.DataFrame({
    "column": df.columns,
    "dtype": df.dtypes.astype(str),
    "missing": df.isna().sum(),
    "missing_pct": (df.isna().mean()*100).round(2),
    "unique": df.nunique()
})
summary

,column,dtype,missing,missing_pct,unique
appointment_id,appointment_id,object,0,0.00,5000
patient_id,patient_id,object,0,0.00,1696
gender,gender,object,0,0.00,3
age,age,int64,0,0.00,63
age_group,age_group,object,0,0.00,6
appointment_type,appointment_type,object,0,0.00,4
booking_date,booking_date,object,0,0.00,593
appointment_date,appointment_date,object,0,0.00,545
appointment_day,appointment_day,object,0,0.00,7
appointment_time,appointment_time,object,0,0.00,3


**Dataset Structure**

The dataset contains 5,000 appointment records across 18 variables, combining patient demographics (gender, age, age_group), appointment details (type, day, time), booking behavior (booking_lead_days), historical patterns (previous_appointments, previous_no_shows), engagement signals (reminder_sent, reminder_channel), and operational factors (distance_to_clinic_km, waiting_time_minutes).

In [6]:
print("Exact duplicate rows:", df.duplicated().sum())
print("Duplicate appointment IDs:", df['appointment_id'].duplicated().sum())
print()
print("Outcome distribution:")
print(df['appointment_outcome'].value_counts())
print()
print(df['appointment_outcome'].value_counts(normalize=True).round(4)*100)

Exact duplicate rows: 0
Duplicate appointment IDs: 0

Outcome distribution:
appointment_outcome
No-Show      2423
Attended     2314
Cancelled     263
Name: count, dtype: int64

appointment_outcome
No-Show      48.46
Attended     46.28
Cancelled     5.26
Name: proportion, dtype: float64


**Duplicates and Target Distribution**

There are no exact duplicate rows and no duplicate appointment IDs — each row represents a unique appointment. The outcome variable splits into three categories: No-Show (2,423, 48.5%), Attended (2,314, 46.3%), and Cancelled (263, 5.3%). This three-way split matters directly for how the target variable should be defined — see Part 4.

In [7]:
print("Missing values:")
print(df.isna().sum()[df.isna().sum() > 0])
print()
# Check whether reminder_channel missingness is random or structural
print("reminder_sent == 'No':", (df['reminder_sent']=='No').sum())
print("reminder_channel missing:", df['reminder_channel'].isna().sum())
print("Do these match?", (df['reminder_sent']=='No').sum() == df['reminder_channel'].isna().sum())

Missing values:
reminder_channel         1366
distance_to_clinic_km      90
waiting_time_minutes       60
dtype: int64

reminder_sent == 'No': 1366
reminder_channel missing: 1366
Do these match? True


**Missing Values — Not Random**

Three columns have missing values: reminder_channel (1,366, 27.3%), distance_to_clinic_km (90, 1.8%), and waiting_time_minutes (60, 1.2%). Critically, reminder_channel's missingness is NOT random — it matches exactly with reminder_sent = 'No' (1,366 in both cases). This means the channel is blank specifically because no reminder was sent at all, not because of a data entry gap. This is a meaningful pattern, not a data quality problem to "fix" — it should be modelled as-is (e.g. a "None" category), not imputed as if it were missing data.

In [8]:
print("Negative booking_lead_days:", (df['booking_lead_days'] < 0).sum())
print("previous_no_shows > previous_appointments (impossible):", (df['previous_no_shows'] > df['previous_appointments']).sum())

Negative booking_lead_days: 0
previous_no_shows > previous_appointments (impossible): 0


**Data Consistency**

No appointments have a negative booking lead time, and no patient has more previous no-shows than previous appointments — both would be logically impossible, and their absence is a good sign the dataset is internally consistent and usable.

# Part 3 (Track Task): Define the Machine Learning Problem

**Proposed Target Variable and Cancellation Handling**

The core business question — can we predict whether a patient will miss their appointment — is naturally a binary classification problem. The proposed target: no_show_target, where No-Show = 1 and Attended = 0.

Cancelled appointments will be excluded from this first model rather than merged into either class. A cancellation is a fundamentally different patient behavior from silently not showing up — a cancellation is a communicated decision, while a no-show is not. Mixing the two would blur what the model is actually trying to predict.

In [9]:
model_df = df[df['appointment_outcome'].isin(['Attended', 'No-Show'])].copy()
model_df['no_show_target'] = (model_df['appointment_outcome'] == 'No-Show').astype(int)

print("Candidate modelling rows:", len(model_df))
print("No-show rate in modelling data:", round(model_df['no_show_target'].mean(), 4))

Candidate modelling rows: 4737
No-show rate in modelling data: 0.5115


**Resulting Modelling Dataset**

Excluding cancellations leaves 4,737 appointments for the initial classifier, with a no-show rate of about 51.2% — a well-balanced target, which is convenient since it means class imbalance is not a major concern for this particular dataset, unlike many real-world no-show datasets.

# Part 4 (Track Task): Potential Input Features

**Candidate Features**

D**emographic:** gender, age. (age_group is derived directly from age and should not be used alongside it — that would be redundant, feeding the model the same information twice in different forms.)

**Appointment characteristics:** appointment_type, appointment_day, appointment_time.

**Booking behavior:** booking_lead_days — how far in advance the appointment was booked.

**Historical behavior:** previous_appointments, previous_no_shows — a patient's own track record. An engineered historical no-show rate (previous_no_shows / previous_appointments) may be worth creating in Week 5.

**Engagement:** reminder_sent, reminder_channel — interpreted together, given the structural missingness identified above.

**Access/operational:** distance_to_clinic_km, waiting_time_minutes — both have limited missing values, subject to confirming they'd actually be known before the appointment happens (see Data Leakage below).

**Excluded as raw features:** appointment_id and patient_id — these are identifiers, not predictive information, and using them directly would let the model "memorize" specific patients rather than learn generalizable patterns.

# Part 5 (Track Task): Initial Modelling Approach

**Initial Modelling Plan**

1. Frame the task as binary supervised classification on non-cancelled appointments.
2. Establish a simple baseline (e.g. a majority-class or DummyClassifier) before evaluating anything more complex — this sets a floor that any real model needs to beat.
3. Build a preprocessing pipeline: impute missing numeric values, encode categorical variables, and handle reminder_channel's structural missingness deliberately (e.g. as its own "No Reminder" category rather than an imputed guess).
4. Start with an interpretable baseline model (Logistic Regression), then compare against tree-based models (Random Forest, Gradient Boosting).
5. Use a stratified train/validation/test split; consider a time-based split instead if the clinic would deploy this model to predict future appointments, since a purely random split could let the model "see the future" during training.
6. Evaluate using recall, precision, F1-score, ROC-AUC, and PR-AUC — not just accuracy, since a naive model could score ~51% accuracy just by guessing the majority class.
7. Check model calibration and explainability before recommending any operational use.

# Part 6 (Required by Brief): Key Modelling Considerations, Risks and Dependencies

**Assumptions, Limitations, Risks and Dependencies**

**Data leakage risk:** waiting_time_minutes may only be known after an appointment happens, not before — if so, it cannot be used as a predictive feature, since it wouldn't be available at prediction time. This needs to be confirmed before Week 5 feature selection.

**Repeated patients:** patient_id appears across multiple appointments. A random train/test split could place the same patient's appointments in both sets, letting the model "cheat" by learning that specific patient's pattern rather than generalizing. A patient-aware or grouped split should be considered.

**Derived/redundant features:** age_group is derived from age; appointment_day is likely derivable from appointment_date. Using both the original and derived version of the same information adds redundancy without adding new signal.

**Structural missingness:** As established above, reminder_channel's missing values are not random — they must be handled with that in mind, not treated as a generic imputation problem.

**Target scope decision:** Excluding cancellations is a deliberate, documented choice — a different valid approach (e.g. a 3-class model) exists and could be revisited later if the business need changes.

**Fairness and responsible use:** Any deployed prediction should support improving service (e.g. proactive reminder calls) rather than being used to penalize or deny care to patients flagged as high-risk.

**Synthetic data limitation:** This dataset is fictional and anonymized — it's suitable for demonstrating sound methodology, but conclusions should not be presented as reflecting real clinical patterns.

**Operational dependency:** A no-show prediction is only useful if the clinic defines a clear point in the process when the prediction would actually be generated and acted upon (e.g. 48 hours before the appointment).

# Part 7: Week 4 Project Summary

**Week 4 Project Summary**


1. Problem addressed: Define the foundation for a machine learning solution that predicts whether a patient will miss (no-show) a scheduled appointment, before it happens.

2. Resources used: HealthConnect_Appointment_Data.csv, HealthConnect_Data_Dictionary.xlsx, and the official Week 4 assignment document.

3. Key observations: The dataset (5,000 rows, 18 columns) is well-structured with no duplicates and only limited, mostly-explainable missingness. The reminder_channel missingness is structural, not random. The no-show rate is close to balanced (~48.5% overall, ~51.2% after excluding cancellations), which simplifies the initial modelling approach.

4. Proposed approach: Frame this as binary classification (No-Show = 1, Attended = 0), excluding Cancelled appointments from the first model. Build a baseline model first, then compare interpretable and tree-based classifiers using leakage-aware, patient-aware validation.

5. Key considerations: Data leakage (particularly waiting_time_minutes), repeated-patient leakage across train/test splits, redundant derived features, and the synthetic nature of the dataset all need to be addressed before any model recommendation.

6. Proposed focus for Week 5: Clean and prepare the modelling dataset, confirm which features are genuinely available before the prediction point, engineer historical no-show rate and any other useful features, and build/evaluate baseline models.